In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder, StandardScaler

import warnings
warnings.filterwarnings('ignore')

In [8]:
df = pd.read_csv('/kaggle/input/datasets/harshtiwari810972/priceperdiction-csv/bengaluru_house_prices (1).csv')


In [9]:
print(df.shape)        # rows and columns
print(df.head())       # first 5 rows
print(df.info())       # data types and nulls
print(df.isnull().sum()) # missing values per column

(13320, 9)
              area_type   availability                  location       size  \
0  Super built-up  Area         19-Dec  Electronic City Phase II      2 BHK   
1            Plot  Area  Ready To Move          Chikka Tirupathi  4 Bedroom   
2        Built-up  Area  Ready To Move               Uttarahalli      3 BHK   
3  Super built-up  Area  Ready To Move        Lingadheeranahalli      3 BHK   
4  Super built-up  Area  Ready To Move                  Kothanur      2 BHK   

   society total_sqft  bath  balcony   price  
0  Coomee        1056   2.0      1.0   39.07  
1  Theanmp       2600   5.0      3.0  120.00  
2      NaN       1440   2.0      3.0   62.00  
3  Soiewre       1521   3.0      1.0   95.00  
4      NaN       1200   2.0      1.0   51.00  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  obj

In [16]:
print(df.columns.tolist())

['area_type', 'availability', 'location', 'size', 'total_sqft', 'bath', 'balcony', 'price']


In [17]:
df.dropna(inplace=True)
print(df.shape)  # check kitni rows bachi


(12710, 8)


In [18]:
# Step 3 - Extract BHK number from size column
df['bhk'] = df['size'].str.extract(r'(\d+)').astype(int)
df.drop(columns=['size'], inplace=True)
print(df.head())

              area_type   availability                  location total_sqft  \
0  Super built-up  Area         19-Dec  Electronic City Phase II       1056   
1            Plot  Area  Ready To Move          Chikka Tirupathi       2600   
2        Built-up  Area  Ready To Move               Uttarahalli       1440   
3  Super built-up  Area  Ready To Move        Lingadheeranahalli       1521   
4  Super built-up  Area  Ready To Move                  Kothanur       1200   

   bath  balcony   price  bhk  
0   2.0      1.0   39.07    2  
1   5.0      3.0  120.00    4  
2   2.0      3.0   62.00    3  
3   3.0      1.0   95.00    3  
4   2.0      1.0   51.00    2  


In [19]:
# Step 4 - Check total_sqft for range values
df[~df['total_sqft'].astype(str).str.replace('.','',1).str.isnumeric()]['total_sqft'].unique()

array(['2100 - 2850', '3067 - 8156', '1042 - 1105', '1145 - 1340',
       '1015 - 1540', '34.46Sq. Meter', '1195 - 1440', '1120 - 1145',
       '3090 - 5002', '1160 - 1195', '1000Sq. Meter', '1115 - 1130',
       '520 - 645', '1000 - 1285', '650 - 665', '633 - 666', '5.31Acres',
       '30Acres', '1445 - 1455', '884 - 1116', '850 - 1093',
       '716Sq. Meter', '547.34 - 827.31', '580 - 650', '3425 - 3435',
       '1804 - 2273', '3630 - 3800', '4000 - 5249', '1500Sq. Meter',
       '142.61Sq. Meter', '1574Sq. Yards', '1250 - 1305', '670 - 980',
       '1005.03 - 1252.49', '1004 - 1204', '645 - 936', '2710 - 3360',
       '2830 - 2882', '596 - 804', '1255 - 1863', '1300 - 1405',
       '117Sq. Yards', '934 - 1437', '980 - 1030', '2249.81 - 4112.19',
       '1070 - 1315', '3040Sq. Meter', '500Sq. Yards', '2806 - 3019',
       '613 - 648', '704 - 730', '1210 - 1477', '3369 - 3464',
       '1125 - 1500', '167Sq. Meter', '1076 - 1199', '381 - 535',
       '524 - 894', '540 - 670', '2725 - 3

In [20]:
# Step 5 - Fix total_sqft column
def convert_sqft(x):
    try:
        if '-' in str(x):
            parts = str(x).split('-')
            return (float(parts[0]) + float(parts[1])) / 2
        return float(x)
    except:
        return None

df['total_sqft'] = df['total_sqft'].apply(convert_sqft)
df.dropna(subset=['total_sqft'], inplace=True)
print(df.shape)
print(df['total_sqft'].head(10))

(12668, 8)
0     1056.0
1     2600.0
2     1440.0
3     1521.0
4     1200.0
5     1170.0
8     1310.0
10    1800.0
11    2785.0
12    1000.0
Name: total_sqft, dtype: float64


In [21]:
# Step 6 - Add price per sqft column
df['price_per_sqft'] = (df['price'] * 100000) / df['total_sqft']
print(df['price_per_sqft'].describe())

count    1.266800e+04
mean     6.876277e+03
std      2.263354e+04
min      2.678298e+02
25%      4.242721e+03
50%      5.376344e+03
75%      7.142857e+03
max      2.300000e+06
Name: price_per_sqft, dtype: float64


In [22]:
# Step 7 - Remove outliers using price_per_sqft per location
def remove_outliers(df):
    df_out = pd.DataFrame()
    for key, subdf in df.groupby('location'):
        m = subdf['price_per_sqft'].mean()
        st = subdf['price_per_sqft'].std()
        reduced = subdf[(subdf['price_per_sqft'] > (m - st)) & 
                        (subdf['price_per_sqft'] <= (m + st))]
        df_out = pd.concat([df_out, reduced], ignore_index=True)
    return df_out

df = remove_outliers(df)
print(df.shape)

(9841, 9)


In [23]:
# Step 8 - Remove rows where bathrooms > bhk + 2
df = df[df['bath'] < df['bhk'] + 2]
print(df.shape)

(9741, 9)


In [24]:
# Step 9 - Drop unnecessary columns
df.drop(columns=['area_type', 'availability'], inplace=True)
print(df.columns.tolist())
print(df.shape)

['location', 'total_sqft', 'bath', 'balcony', 'price', 'bhk', 'price_per_sqft']
(9741, 7)


In [35]:
# Step 11 - Label low frequency locations as 'other'
location_stats_less10 = location_stats[location_stats <= 10]
df['location'] = df['location'].apply(lambda x: 'other' if x in location_stats_less10 else x)

In [25]:
# Step 10 - Handle location column
df['location'] = df['location'].apply(lambda x: x.strip())
location_stats = df.groupby('location')['location'].agg('count').sort_values(ascending=False)
print(location_stats.head(10))
print()
print('Total unique locations:', len(location_stats))

location
Whitefield                  507
Sarjapur  Road              294
Electronic City             285
Kanakpura Road              190
Uttarahalli                 179
Yelahanka                   179
Raja Rajeshwari Nagar       162
Thanisandra                 154
Marathahalli                143
Electronic City Phase II    125
Name: location, dtype: int64

Total unique locations: 785


In [44]:
# Step 11 - Label low frequency locations as 'other'
location_stats_less10 = location_stats[location_stats <= 10]
df['location'] = df['location'].apply(lambda x: 'other' if x in location_stats_less10 else x)

print('Unique locations after:', df['location'].nunique())

Unique locations after: 185


In [47]:
# Final check
print(df.shape)
print(df.info())
print(df.head())

(9741, 7)
<class 'pandas.core.frame.DataFrame'>
Index: 9741 entries, 0 to 9840
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   location        9741 non-null   object 
 1   total_sqft      9741 non-null   float64
 2   bath            9741 non-null   float64
 3   balcony         9741 non-null   float64
 4   price           9741 non-null   float64
 5   bhk             9741 non-null   int64  
 6   price_per_sqft  9741 non-null   float64
dtypes: float64(5), int64(1), object(1)
memory usage: 608.8+ KB
None
              location  total_sqft  bath  balcony  price  bhk  price_per_sqft
0                other      1100.0   2.0      1.0   70.0    2     6363.636364
1                other      1672.0   3.0      2.0  150.0    3     8971.291866
2                other      1750.0   3.0      3.0  149.0    3     8514.285714
3                other      1750.0   3.0      2.0  150.0    3     8571.428571
4  Devarachikkanahalli   

In [48]:
# Step 12 - Prepare X and y
X = df.drop(columns=['price', 'price_per_sqft'])
y = df['price']

print('X shape:', X.shape)
print('y shape:', y.shape)
print(X.head())

X shape: (9741, 5)
y shape: (9741,)
              location  total_sqft  bath  balcony  bhk
0                other      1100.0   2.0      1.0    2
1                other      1672.0   3.0      2.0    3
2                other      1750.0   3.0      3.0    3
3                other      1750.0   3.0      2.0    3
4  Devarachikkanahalli      1250.0   2.0      3.0    3


In [49]:
# Step 13 - One Hot Encoding for location
X = pd.get_dummies(X, columns=['location'])
print(X.shape)
print(X.head())

(9741, 189)
   total_sqft  bath  balcony  bhk  location_1st Phase JP Nagar  \
0      1100.0   2.0      1.0    2                        False   
1      1672.0   3.0      2.0    3                        False   
2      1750.0   3.0      3.0    3                        False   
3      1750.0   3.0      2.0    3                        False   
4      1250.0   2.0      3.0    3                        False   

   location_2nd Stage Nagarbhavi  location_5th Phase JP Nagar  \
0                          False                        False   
1                          False                        False   
2                          False                        False   
3                          False                        False   
4                          False                        False   

   location_6th Phase JP Nagar  location_7th Phase JP Nagar  \
0                        False                        False   
1                        False                        False   
2          

In [50]:
# Step 14 - Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)

X_train shape: (7792, 189)
X_test shape: (1949, 189)


In [51]:
# Step 15 - Train Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

print('Model training complete!')

Model training complete!


In [52]:
# Step 16 - Evaluate Model
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('R2 Score  :', round(r2, 4))
print('MAE       :', round(mae, 2))
print('RMSE      :', round(rmse, 2))

R2 Score  : 0.6841
MAE       : 24.2
RMSE      : 48.78


In [53]:
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import cross_val_score

# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
y_pred_ridge = ridge.predict(X_test)
print('Ridge R2:', round(r2_score(y_test, y_pred_ridge), 4))

# Lasso Regression
lasso = Lasso(alpha=1.0)
lasso.fit(X_train, y_train)
y_pred_lasso = lasso.predict(X_test)
print('Lasso R2:', round(r2_score(y_test, y_pred_lasso), 4))

# Cross Validation on Linear Regression
scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')
print('Cross Val R2 scores:', np.round(scores, 4))
print('Mean R2:', round(scores.mean(), 4))

Ridge R2: 0.6825
Lasso R2: 0.6158
Cross Val R2 scores: [0.5013 0.6053 0.6238 0.615  0.6927]
Mean R2: 0.6076


In [55]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print('Decision Tree R2:', round(r2_score(y_test, y_pred_dt), 4))

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('Random Forest R2:', round(r2_score(y_test, y_pred_rf), 4))

Decision Tree R2: 0.4262
Random Forest R2: 0.6069


In [57]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
print('Gradient Boosting R2:', round(r2_score(y_test, y_pred_gb), 4))
print('Gradient Boosting MAE:', round(mean_absolute_error(y_test, y_pred_gb), 2))

Gradient Boosting R2: 0.5928
Gradient Boosting MAE: 22.78


In [58]:
# Step 17 - Prediction Function
def predict_price(location, total_sqft, bath, balcony, bhk):
    loc_index = np.where(X.columns == 'location_' + location)[0]
    
    x = np.zeros(len(X.columns))
    x[0] = total_sqft
    x[1] = bath
    x[2] = balcony
    x[3] = bhk
    
    if len(loc_index) > 0:
        x[loc_index[0]] = 1

    return round(model.predict([x])[0], 2)

# Test
print(predict_price('Whitefield', 1500, 2, 1, 3))
print(predict_price('Sarjapur  Road', 1200, 2, 1, 2))
print(predict_price('Rajaji Nagar', 1000, 2, 1, 2))

94.96
64.07
211.62


In [59]:
import pickle

# Model save karo
with open('house_price_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Columns save karo
with open('columns.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)

print('Model saved successfully!')

Model saved successfully!
